# SpinGlassPEPS

W tej sekcji pokażemy zastosowanie sieci tensorowych do optymalizacji i próbkowania z modelu isinga na przykładzie algorytmu SpinGlassPEPS. Jest to bardzo ogólny algorytm który posiada wiele dodatkowych funkcjonalności, takich jak wykrywanie dropletów czy analiza degeneracji stanu podstawowego.

## Model Pottsa

W modelu Isinga każdy spin przyjmuje jedną z dwóch możliwych wartości, co czyni go szczególnym przypadkiem bardziej ogólnego modelu Pottsa, w którym spin może znajdować się w jednym z $p$ stanów. Przejście od modelu Isinga ($p=2$) do modelu Pottsa o większej liczbie stanów można zrealizować poprzez grupowanie spinów tj. traktując kilka sąsiednich spinów jako jeden wierzchołek (zmienna Pottsa) o liczbie stanów równej $2^d$, gdzie $d$ to liczba spinów w grupie.

Takie grupowanie jest szczególnie użyteczne w kontekście sieci tensorowych, ponieważ pozwala na redukcję wymiarowości problemu poprzez przeniesienie części złożoności do większej liczby stanów pojedynczego węzła, zamiast utrzymywania bardzo dużej liczby połączeń pomiędzy wieloma węzłami o małej liczbie stanów. W konsekwencji, struktura sieci tensorowej staje się prostsza, co może ułatwiać kompresję, przyspieszać obliczenia oraz poprawiać stabilność numeryczną w procesach kontrak


Niech $\hat{N}$ będzie ilością zmiennych Pottsa $x_n$, $n \in [\hat{N}]$ oraz niech $\mathcal{F}$ będzie grafem utworzonym przez zamianę modelu isinga na model Pottsa. Niech $x$ oznacza konfiguracje zmiennych Pottsa. Wtedy enenergię konfiguracji definiujemy jako:

$$
H(x) = \sum_{(m, n) \in \mathcal{F}} E_{x_n x_m} + \sum_{n=1}^{\hat{N}} E_{x_{n}}.
$$

Tutaj $E_{x_n}$ oznacza energię wierzchołka $x_n$, a $E_{x_n x_m}$ oznacza energię interakcji pomiędzy wierzchołkami.


## Zamiana dwave na hamiltonian potsa

![image](pictures/tn_dw.png)

Grupujemy spiny w każdej komórce elementarnej, uzyskując uogólniony Hamiltonian Pottsa, zdefiniowany na siatce kwadratowej z oddziaływaniami między najbliższymi sąsiadami (niebieskie linie) oraz sprzężeniami po przekątnych (zielone linie), jak pokazano na panelach (b) i (d).

Przy takim grupowaniu zmienne potsa mogą przyjmować wartości od 1 do $2^d$ gdzie $d$ jest ilością spinów w komórce elementarnej. Łatwo zauważyć że każdej wartości zmiennej Pottsa można przypisać konkretną kombinacje spinów. 

## Schemat działania algorytmu

![image](pictures/alg_software-1.png)

Problem Isinga (a) jest przekształcany w Hamiltonian Pottsa określony na grafie króla (b). Dzięki temu funkcja podziału tego Hamiltonianu może zostać przedstawiona jako tensorowa sieć PEPS na kwadratowej siatce (c). Główny algorytm wykonuje przeszukiwanie typu branch and bound w przestrzeni prawdopodobieństw, stopniowo budując najbardziej prawdopodobne konfiguracje poprzez dodawanie po jednej zmiennej Pottsa. Warunkowe prawdopodobieństwa brzegowe uzyskiwane są przez przybliżone zwęrzanie odpowiadającej im sieci tensorowej (d). Pełny przebieg branch and bound daje w efekcie kandydata na najbardziej prawdopodobną konfigurację (stan podstawowy) (e), wraz ze zlokalizowanymi wzbudzeniami nałożonymi na nią (f).

# Przykłady

## Model isinga na grafie króla


In [1]:
using SpinGlassPEPS
using Logging

disable_logging(LogLevel(1))

function get_kings_graph_instance(topology::NTuple{3, Int})
    m, n, t = topology
    "$(@__DIR__)/instancje/$(m)x$(n)x$(t).txt"
end


get_kings_graph_instance (generic function with 1 method)

In [2]:
# ustawienie parametrów i wczytanie instancji
topology = (3, 3, 2)
m, n, t = topology
T = Float32

instance = get_kings_graph_instance(topology)

# ustawiamy rodzaj siatki w sieci tensorowej. Ta jest dla grafów króla 
lattice = super_square_lattice(topology) 


Dict{Int64, Tuple{Int64, Int64}} with 18 entries:
  5  => (1, 3)
  16 => (3, 2)
  7  => (2, 1)
  12 => (2, 3)
  8  => (2, 1)
  17 => (3, 3)
  1  => (1, 1)
  13 => (3, 1)
  15 => (3, 2)
  4  => (1, 2)
  11 => (2, 3)
  2  => (1, 1)
  10 => (2, 2)
  9  => (2, 2)
  6  => (1, 3)
  18 => (3, 3)
  14 => (3, 1)
  3  => (1, 2)

In [3]:
# Budujemy hamiltonian Pottsa

potts_h = potts_hamiltonian(
    ising_graph(instance),
    spectrum = full_spectrum,
    cluster_assignment_rule = lattice,
)

LabelledGraphs.LabelledGraph{MetaGraphs.MetaDiGraph{Int64, Float64}, Tuple{Int64, Int64}}([(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)], {9, 20} directed Int64 metagraph with Float64 weights defined by :weight (default weight 1.0), Dict((3, 2) => 8, (1, 2) => 2, (3, 1) => 7, (1, 1) => 1, (3, 3) => 9, (1, 3) => 3, (2, 2) => 5, (2, 1) => 4, (2, 3) => 6))

In [4]:
# Ustawiamy parametry algorytmu

params = MpsParameters{T}(; bond_dim = 16, num_sweeps = 1)
search_params = SearchParameters(; max_states = 2^8, cutoff_prob = 1E-4)

SearchParameters(256, 0.0001)

## Obroty

Warto zwrócić uwagę, że heurystyczne zwęrzanie sieci tensorowej często wprowadza pewne obciążenie wyników. By się go pozbyć robimy obroty.

In [5]:
 best_energies = []

for transform ∈ all_lattice_transformations
        net = PEPSNetwork{KingSingleNode{GaugesEnergy}, Dense, T}(
            m, n, potts_h, transform,
        )

        ctr = MpsContractor(SVDTruncate, net, params; 
            onGPU = false, beta = T(2), graduate_truncation = true,
        )

        droplets = SingleLayerDroplets(; max_energy = 10, min_size = 5, metric = :hamming)
        merge_strategy = merge_branches(
            ctr; merge_prob = :none , droplets_encoding = droplets,
        )

        sol, _ = low_energy_spectrum(ctr, search_params, merge_strategy)
        droplets = unpack_droplets(sol, T(2))
        ig_states = decode_potts_hamiltonian_state.(Ref(potts_h), droplets.states)
        ldrop = length(droplets.states)

        println("Best energy for transform $(transform) is $(sol.energies[1])")
        println("Number of droplets for transform $(transform) is $(ldrop)")
        println("------------------")

        push!(best_energies, sol.energies[1])
        clear_memoize_cache()
        
    end
    println("Best energy found: $(minimum(best_energies))")


Preprocessing: 100%|████████████████████████████████████| Time: 0:00:25
Search: 100%|███████████████████████████████████████████| Time: 0:00:07


Best energy for transform LatticeTransformation((1, 2, 3, 4), false) is -19.933226
Number of droplets for transform LatticeTransformation((1, 2, 3, 4), false) is 14
------------------
Best energy for transform LatticeTransformation((4, 1, 2, 3), true) is -19.933226
Number of droplets for transform LatticeTransformation((4, 1, 2, 3), true) is 16
------------------
Best energy for transform LatticeTransformation((3, 4, 1, 2), false) is -19.933228
Number of droplets for transform LatticeTransformation((3, 4, 1, 2), false) is 16
------------------
Best energy for transform LatticeTransformation((2, 3, 4, 1), true) is -19.933226
Number of droplets for transform LatticeTransformation((2, 3, 4, 1), true) is 12
------------------
Best energy for transform LatticeTransformation((4, 3, 2, 1), false) is -19.933226
Number of droplets for transform LatticeTransformation((4, 3, 2, 1), false) is 14
------------------
Best energy for transform LatticeTransformation((2, 1, 4, 3), false) is -19.933228
N

## Mały Pegasus

In [6]:
using CUDA

onGPU = CUDA.has_cuda_gpu()

function run_pegasus_bench(::Type{T}; topology::NTuple{3, Int}) where {T}
    m, n, t = topology
    instance = "$(@__DIR__)/instancje/P4_CBFM-P.txt"
    results_folder = "$(@__DIR__)/lbp"
    isdir(results_folder) || mkdir(results_folder)

    lattice = pegasus_lattice(topology)

    potts_h = potts_hamiltonian(
        ising_graph(instance),
        spectrum = full_spectrum,
        cluster_assignment_rule = lattice,
    )

    potts_h = truncate_potts_hamiltonian(potts_h, T(2), 2^16, results_folder, "P4_CBFM-P"; tol=1e-6, iter=2)

    params = MpsParameters{T}(bond_dim=16, num_sweeps=1)
    search_params = SearchParameters(max_states=2^10, cutoff_prob=1e-4)

    best_energies = T[]

    for transform in all_lattice_transformations
        net = PEPSNetwork{SquareCrossDoubleNode{GaugesEnergy}, Sparse, T}(m, n, potts_h, transform)
        ctr = MpsContractor(Zipper, net, params; onGPU=onGPU, beta=T(1), graduate_truncation=true)

        droplets = SingleLayerDroplets(max_energy=10, min_size=54, metric=:hamming)
        merge_strategy = merge_branches(ctr; merge_prob=:none, droplets_encoding=droplets)

        sol, _ = low_energy_spectrum(ctr, search_params, merge_strategy)
        sol2 = unpack_droplets(sol, T(2))
        ldrop = length(sol2.states)

        println("Best energy for transform $(transform) is $(sol.energies[1])")
        println("Number of droplets for transform $(transform) is $(ldrop)")
        println("------------------")


        push!(best_energies, sol.energies[1])
        clear_memoize_cache()
    end

    ground = best_energies[1]
     println("Best energy found: $(minimum(best_energies))")
    
    
end


@time run_pegasus_bench(T; topology = (3, 3, 3))

Preprocessing: 100%|████████████████████████████████████| Time: 0:01:17
Search: 100%|███████████████████████████████████████████| Time: 0:00:48


Best energy for transform LatticeTransformation((1, 2, 3, 4), false) is -469.0
Number of droplets for transform LatticeTransformation((1, 2, 3, 4), false) is 2
------------------


Preprocessing: 100%|████████████████████████████████████| Time: 0:00:20
Search: 100%|███████████████████████████████████████████| Time: 0:00:38


Best energy for transform LatticeTransformation((4, 1, 2, 3), true) is -469.0
Number of droplets for transform LatticeTransformation((4, 1, 2, 3), true) is 1
------------------


Preprocessing: 100%|████████████████████████████████████| Time: 0:00:10
Search: 100%|███████████████████████████████████████████| Time: 0:00:49


...

Best energy for transform LatticeTransformation((2, 1, 4, 3), false) is -469.0
Number of droplets for transform LatticeTransformation((2, 1, 4, 3), false) is 2
------------------


Preprocessing: 100%|████████████████████████████████████| Time: 0:00:12
Search: 100%|███████████████████████████████████████████| Time: 0:00:35


Best energy for transform LatticeTransformation((1, 4, 3, 2), true) is -469.0
Number of droplets for transform LatticeTransformation((1, 4, 3, 2), true) is 3
------------------


Preprocessing: 100%|████████████████████████████████████| Time: 0:00:09
Search: 100%|███████████████████████████████████████████| Time: 0:00:48


Best energy for transform LatticeTransformation((3, 2, 1, 4), true) is -469.0
Number of droplets for transform LatticeTransformation((3, 2, 1, 4), true) is 1
------------------
Best energy found: -469.0
560.741185 seconds (1.59 G allocations: 113.053 GiB, 5.03% gc time, 105 lock conflicts, 20.34% compilation time: 2% of which was recompilation)


## Wypełnianie obrazka (wymaga GPU)

In [7]:
using SpinGlassPEPS
using CUDA
using Pkg
using Logging

# for visualisation of results, we need following packages
try
    using Colors, Images
catch e
    if isa(e, ArgumentError) || isa(e, LoadError)
        Pkg.add("Colors")
        Pkg.add("Images")
    else
        rethrow(e)
    end
end

disable_logging(LogLevel(1))

instance = "$(@__DIR__)/instancje/triplepoint4-plain-ring.h5"
onGPU = CUDA.has_cuda_gpu()

GEOMETRY = SquareSingleNode
LAYOUT = GaugesEnergy
SPARSITY = Sparse
STRATEGY = SVDTruncate
GAUGE =  NoUpdate

function bench_inpaining(::Type{T}, β::Real, max_states::Integer, bond_dim::Integer) where {T}
	potts_h= potts_hamiltonian(instance, 120, 120)

	params = MpsParameters{T}(; bond_dim = bond_dim, method = :svd)
	search_params = SearchParameters(; max_states = max_states)
	net = PEPSNetwork{GEOMETRY{LAYOUT}, SPARSITY, T}(120, 120, potts_h, rotation(0))
	ctr = MpsContractor{STRATEGY, GAUGE, T}(net, params; onGPU = onGPU, beta = convert(Float64, β), graduate_truncation = true)
    droplets = SingleLayerDroplets(; max_energy = 100, min_size = 100 , metric = :hamming, mode=:RMF)
	merge_strategy = merge_branches(ctr; merge_prob = :none, droplets_encoding = droplets)

	sol, info = low_energy_spectrum(ctr, search_params, merge_strategy)
    ground = sol.energies[begin]

    println("Best energy found: $(ground)")
    sol
end

function visualize_result(sol::Solution)
    solution = sol.states[begin]
    solution = reshape(solution, (120, 120))

    sol2 = unpack_droplets(sol, 6)
    droplet_state = sol2.states[2]
    droplet_state = reshape(droplet_state, (120, 120))

    droplet = findall(solution .!= droplet_state)

    color_map = Dict(
        1 => RGB(225/255, 0.0, 100/255),
        2 => RGB(100/255, 225/255, 0.0),
        3 => RGB(0/255, 100/255, 225/255),
        4 => RGB(1.0, 1.0, 1.0)
    )

    img = zeros(RGB, 120, 120)

    for i in 1:120
        for j in 1:120
            img[j, i] = color_map[solution[i, j]]
        end
    end

    save(joinpath("$(@__DIR__)/inpaining_solution.png"), img)
    println("Solution visualisation saved to $(@__DIR__)/inpaining_solution.png")
    
    for (i,j) in Tuple.(droplet)
        img[j, i] = RGB(255/255, 255/255, 0/255)
    end

    save(joinpath("$(@__DIR__)/inpaining_droplet.png"), img)
    println("Droplet visualisation saved to $(@__DIR__)/inpaining_droplet.png")
end


sol = bench_inpaining(Float64, 6, 64, 4)
visualize_result(sol)

Preprocessing: 100%|████████████████████████████████████| Time: 0:01:14
Search:  67%|█████████████████████████████              |  ETA: 0:06:36KExcessive output truncated after 524407 bytes.